In [1]:
import plotly.graph_objects as go

steps = [
    ("Absolute<br>Rundenzeit",                                      "absolute", 89.0),
    ("− Streckenbasis<br><sub>gleiche Strecke</sub>",               "relative", -78.0),
    ("− Konstrukteur<br><sub>gleiches Auto</sub>",                  "relative",  -3.0),
    ("− Fahrer-Können<br><sub>gleicher Fahrer</sub>",               "relative",  -2.0),
    ("− Spritlast<br><sub>≈ konstant über 1 Stint</sub>",           "relative",  -3.0),
    ("− Wetter / Track Evo<br><sub>≈ konstant über 1 Stint</sub>",  "relative",  -1.5),
    ("Delta-Zeit<br><sub>= Reifenabnutzung</sub>",                  "total",      1.5),
]

xs       = [s[0] for s in steps]
measures = [s[1] for s in steps]
ys       = [s[2] for s in steps]
texts    = [f"{ys[0]:.1f} s"] + [f"{y:+.1f} s" for y in ys[1:-1]] + [f"{ys[-1]:.1f} s"]

noise_color  = "#64748b"
signal_color = "#f59e0b"

fig = go.Figure(go.Waterfall(
    orientation="v",
    measure=measures,
    x=xs,
    y=ys,
    text=texts,
    textposition="outside",
    textfont=dict(size=12, color="#1e293b"),
    connector={"line": {"color": "#cbd5e1", "dash": "dot", "width": 1}},
    decreasing={"marker": {"color": noise_color, "line": {"color": "white", "width": 1.5}}},
    increasing={"marker": {"color": noise_color}},
    totals    ={"marker": {"color": signal_color, "line": {"color": "white", "width": 1.5}}},
    hovertemplate="<b>%{x}</b><br>%{y:+.2f} s<extra></extra>",
))

fig.update_layout(
    title=dict(
        text=(
            "<b>Von der absoluten Rundenzeit zur Delta-Zeit</b>"
            "<br><sub>Im selben Stint wirken Strecke, Auto, Fahrer und (näherungsweise) "
            "Sprit & Wetter auf Referenz- und aktuelle Runde identisch — die "
            "<b style='color:#475569'>Confounder kürzen sich heraus</b>. "
            "Übrig bleibt, was sich über den Stint verändert: "
            f"<b style='color:{signal_color}'>Reifenabnutzung</b>.</sub>"
        ),
        x=0.5, xanchor="center",
    ),
    yaxis=dict(title="Sekunden", gridcolor="#e2e8f0", rangemode="tozero"),
    xaxis=dict(tickangle=0),
    template="plotly_white",
    height=620,
    margin=dict(t=140, b=120, l=70, r=40),
    showlegend=False,
)

fig.add_annotation(
    x=xs[-1], y=ys[-1], xref="x", yref="y",
    text="<b>Signal für das Modell</b>",
    showarrow=False, yshift=48,
    font=dict(size=13, color=signal_color),
)

fig.show()

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

components = [
    ("Individual Track",       78.0, 0.00),
    ("Car Performance",         3.0, 0.00),
    ("Drivers Performance",        2.0, 0.10),
    ("Fuel",            3.0, 0.15),
    ("Weather / Track Evolution",   1.5, 0.05),
    ("Tire wear ",      1.5, 1.20),
]
signal_idx = 5

noise_palette = ["#505358", "#760060", "#690023", "#0a6800", "#0c006b"]
signal_color  = "#f59e0b"
colors        = noise_palette + [signal_color]

names        = [c[0] for c in components]
abs_values   = [c[1] for c in components]
delta_values = [c[2] for c in components]

abs_total   = sum(abs_values)
delta_total = sum(delta_values)
abs_signal_pct   = abs_values[signal_idx]   / abs_total   * 100
delta_signal_pct = delta_values[signal_idx] / delta_total * 100

delta_mask     = [v > 0 for v in delta_values]
delta_names    = [n for n, keep in zip(names,  delta_mask) if keep]
delta_filtered = [v for v, keep in zip(delta_values, delta_mask) if keep]
delta_colors   = [c for c, keep in zip(colors, delta_mask) if keep]

pull_signal = 0.08
pull_abs   = [0.0]*len(components);      pull_abs[signal_idx] = pull_signal
pull_delta = [0.0]*len(delta_filtered);  pull_delta[-1]       = pull_signal

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "domain"}, {"type": "domain"}]],
    subplot_titles=(
        f"<b>Absolute lap time</b><br><sub>Σ ≈ {abs_total:.0f} s</sub>",
        f"<b>Delta lap time</b><br><sub>Σ ≈ {delta_total:.2f} s</sub>",
    ),
    horizontal_spacing=0.05,
)

fig.add_trace(go.Pie(
    labels=names, values=abs_values,
    marker=dict(colors=colors, line=dict(color="white", width=2)),
    hole=0.6, sort=False, direction="clockwise",
    pull=pull_abs,
    textinfo="percent",
    textfont=dict(color="white", size=12),
    hovertemplate="<b>%{label}</b><br>%{value:.2f} s (%{percent})<extra></extra>",
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=delta_names, values=delta_filtered,
    marker=dict(colors=delta_colors, line=dict(color="white", width=2)),
    hole=0.6, sort=False, direction="clockwise",
    pull=pull_delta,
    textinfo="percent",
    textfont=dict(color="white", size=12),
    hovertemplate="<b>%{label}</b><br>%{value:.2f} s (%{percent})<extra></extra>",
), row=1, col=2)

fig.update_layout(
    title=dict(
        text=(
            "Predicting <b>absoulte lap time</b> vs. <b>delta lap time</b>"
            "<br><sub>The abolute lap time has a lot of noise "
            f"while the delta lap time is dominated by the <b style='color:{signal_color}'>tire wear</b>.</sub>"
        ),
        x=0.5, y=0.94, xanchor="center",
    ),
    template="plotly_white",
    height=580,
    legend=dict(orientation="h", y=-0.08, xanchor="center", x=0.5,
                title="<b>Component</b>"),
    margin=dict(t=140, b=80, l=40, r=40),
)

# Mittelpunkte aus den tatsaechlich vergebenen Subplot-Domains ableiten,
# damit der Text exakt im Donut-Loch sitzt.
def domain_center(dom):
    return (dom[0] + dom[1]) / 2

c_abs   = fig.data[0].domain
c_delta = fig.data[1].domain

fig.add_annotation(
    text=f"<b style='font-size:26px;color:{signal_color}'>{abs_signal_pct:.1f}%</b>"
         f"<br><span style='color:#475569'>Tire wear</span>",
    x=domain_center(c_abs.x), y=domain_center(c_abs.y),
    xref="paper", yref="paper",
    showarrow=False, align="center", xanchor="center", yanchor="middle",
)
fig.add_annotation(
    text=f"<b style='font-size:26px;color:{signal_color}'>{delta_signal_pct:.0f}%</b>"
         f"<br><span style='color:#475569'>Tire wear</span>",
    x=domain_center(c_delta.x), y=domain_center(c_delta.y),
    xref="paper", yref="paper",
    showarrow=False, align="center", xanchor="center", yanchor="middle",
)

fig.show()

In [3]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

rng = np.random.default_rng(7)

stints = [
    {"label": "Monaco · Driver A",      "baseline": 76.4, "color": "#3b82f6"},
    {"label": "Monza · Driver B",       "baseline": 84.9, "color": "#10b981"},
    {"label": "Silverstone · Driver C", "baseline": 91.7, "color": "#ec4899"},
    {"label": "Spa · Driver D",         "baseline": 105.3,"color": "#8b5cf6"},
]

stint_length = 20
wear_per_lap = 0.11
ages = np.arange(1, stint_length + 1)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "<b>Absolute lap time</b>",
        "<b>Delta lap time</b>",
    ),
    horizontal_spacing=0.11,
)

for s in stints:
    noise = rng.normal(0, 0.18, stint_length)
    laps = s["baseline"] + wear_per_lap * (ages - 1) + noise
    deltas = laps - laps[0]

    fig.add_trace(go.Scatter(
        x=ages, y=laps,
        mode="lines+markers",
        name=s["label"],
        line=dict(color=s["color"], width=3.5),
        marker=dict(size=11, line=dict(color="white", width=1.6)),
        legendgroup=s["label"], showlegend=True,
        hovertemplate=f"<b>{s['label']}</b><br>Runde im Stint: %{{x}}<br>Rundenzeit: %{{y:.2f}} s<extra></extra>",
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=ages, y=deltas,
        mode="lines+markers",
        name=s["label"],
        line=dict(color=s["color"], width=3.5),
        marker=dict(size=11, line=dict(color="white", width=1.6)),
        legendgroup=s["label"], showlegend=False,
        hovertemplate=f"<b>{s['label']}</b><br>Runde im Stint: %{{x}}<br>Delta: %{{y:+.2f}} s<extra></extra>",
    ), row=1, col=2)

fig.update_layout(
    title=dict(
        text=(
            "<b>Looking at the absolute lap time makes the tracks more comparable</b>"
            "<br><sub>The lap time is dominated by the time to complete one lap "
            "while the actual information is in the delta time.</sub>"
        ),
        x=0.5, y=0.96, xanchor="center",
        font=dict(size=30),
    ),
    template="plotly_white",
    width=1800,
    height=720,
    legend=dict(
        orientation="h", y=-0.16, xanchor="center", x=0.5,
        bgcolor="rgba(0,0,0,0)",
        font=dict(size=20),
    ),
    margin=dict(t=150, b=120, l=110, r=60),
    font=dict(family="Inter, system-ui, sans-serif", size=20, color="#1e293b"),
)

# Subplot-Titel groesser
for ann in fig.layout.annotations:
    ann.font.size = 24

axis_style = dict(gridcolor="#eef2f7", zerolinecolor="#cbd5e1",
                  showline=True, linecolor="#cbd5e1", ticks="outside", tickcolor="#cbd5e1",
                  title_font=dict(size=22), tickfont=dict(size=18))

fig.update_xaxes(title_text="Lap in a stint", row=1, col=1, **axis_style)
fig.update_xaxes(title_text="Lap in a stint", row=1, col=2, **axis_style)
fig.update_yaxes(title_text="Lap time [s]",       row=1, col=1, **axis_style)
fig.update_yaxes(title_text="Delta to reference [s]", row=1, col=2, **axis_style)

fig.show()